# Graphing a Water Balance — a Python Port of Tony Ladson's R Method

**Companion notebook to:** _Graphing a Water Balance — a Python Port of Tony Ladson's R Method_

**Source:** [Graphing a water balance](https://tonyladson.wordpress.com/2017/08/15/graphing-a-water-balance/) — Tony Ladson, 15 August 2017. R source: [gist.github.com/TonyLadson/4d42e2cedc20aa1ff04a06631af88551](https://gist.github.com/TonyLadson/4d42e2cedc20aa1ff04a06631af88551) (`Water_balance_waterfall.R`).

This is a Python port of Ladson's waterfall-chart construction for visualising an urban catchment water balance, using the same real published data he uses: two example periods (driest and wettest) from Mitchell, V.G., McMahon, T.A. & Mein, R.G. (2003), "Components of the Total Water Balance of an Urban Catchment," *Environmental Management* 32(6): 735-746.

**A genuine finding, not part of the original R script:** checking whether each dataset's flux terms actually sum to the reported "change in storage" value turns up something worth knowing before trusting a waterfall chart on face value — see Section 3.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
print('numpy:', np.__version__)

numpy: 2.4.6


## 1. The data

As transcribed in Ladson's gist, citing Mitchell et al. (2003).

In [2]:
driest = {"Precipitation": 247, "Mains": 269, "Evapotranspiration": -347,
          "Stormwater": -74, "Wastewater": -107, "Change in storage": 12}
wettest = {"Precipitation": 914, "Mains": 141, "Evapotranspiration": -605,
           "Stormwater": -290, "Wastewater": -126, "Change in storage": 34}

for name, wb in [('Driest', driest), ('Wettest', wettest)]:
    print(f'{name}: {wb}')

Driest: {'Precipitation': 247, 'Mains': 269, 'Evapotranspiration': -347, 'Stormwater': -74, 'Wastewater': -107, 'Change in storage': 12}
Wettest: {'Precipitation': 914, 'Mains': 141, 'Evapotranspiration': -605, 'Stormwater': -290, 'Wastewater': -126, 'Change in storage': 34}


## 2. Waterfall geometry

Ladson's R builds the bar geometry with a running cumulative sum, then **forces the final bar (Change in storage) to end at exactly zero** regardless of what the raw cumulative sum produces — a charting convention that always visually closes the diagram:

```r
# Ladson's R
wb_end <- cumsum(amount)
wb_end <- replace(wb_end, length(wb_end), 0)
wb_start <- lag(wb_end)
wb_start <- replace(wb_start, 1, 0)
```

In [3]:
def waterfall_geometry(wb):
    """Reproduce Ladson's R cumsum/lag construction for waterfall bar geometry."""
    terms = list(wb.keys())
    amounts = np.array(list(wb.values()), dtype=float)
    wb_end = np.cumsum(amounts)
    wb_end[-1] = 0.0  # force the final bar to close at zero, matching the R script
    wb_start = np.roll(wb_end, 1)
    wb_start[0] = 0.0
    kinds = ['storage' if t == 'Change in storage' else ('in' if a > 0 else 'out')
             for t, a in zip(terms, amounts)]
    return terms, amounts, wb_start, wb_end, kinds

terms, amounts, wb_start, wb_end, kinds = waterfall_geometry(driest)
for t, a, s, e, k in zip(terms, amounts, wb_start, wb_end, kinds):
    print(f'{t:22s} amount={a:+6.0f}  bar=({s:7.1f} -> {e:7.1f})  kind={k}')

Precipitation          amount=  +247  bar=(    0.0 ->   247.0)  kind=in
Mains                  amount=  +269  bar=(  247.0 ->   516.0)  kind=in
Evapotranspiration     amount=  -347  bar=(  516.0 ->   169.0)  kind=out
Stormwater             amount=   -74  bar=(  169.0 ->    95.0)  kind=out
Wastewater             amount=  -107  bar=(   95.0 ->   -12.0)  kind=out
Change in storage      amount=   +12  bar=(  -12.0 ->     0.0)  kind=storage


## 3. Does the balance actually close? (not in the original script)

The chart construction above *always* draws the last bar closing at zero — that's a property of the chart, not a check on the data. It's worth asking separately whether the five real flux terms (Precipitation + Mains − Evapotranspiration − Stormwater − Wastewater) actually sum to the *reported* "Change in storage" value, since the water balance equation says they should.

In [4]:
for name, wb in [('Driest', driest), ('Wettest', wettest)]:
    terms_list = list(wb.items())
    flux_terms = terms_list[:-1]
    reported = terms_list[-1][1]
    total_flux = sum(v for _, v in flux_terms)
    discrepancy = total_flux - reported
    status = 'closes exactly' if discrepancy == 0 else f'does NOT close ({discrepancy:+.0f} mm gap)'
    print(f'{name:8s} sum of 5 flux terms = {total_flux:+4.0f} mm, reported storage change = {reported:+3.0f} mm -> {status}')

Driest   sum of 5 flux terms =  -12 mm, reported storage change = +12 mm -> does NOT close (-24 mm gap)
Wettest  sum of 5 flux terms =  +34 mm, reported storage change = +34 mm -> closes exactly


The wettest period closes exactly. The driest period doesn't — there's a real 24 mm gap between what the five measured/estimated flux terms sum to and the reported change in storage. That's not a flaw in Ladson's chart (the chart isn't claiming otherwise; it just draws the terms in sequence and anchors the last bar at zero by construction) — it's a genuine, unremarked feature of the underlying academic data, and a reasonable reminder that "change in storage" in a real observational water balance often functions partly as a residual/catch-all term rather than something independently measured to the same precision as the other components. Worth knowing before reading too much precision into any single waterfall chart.

## 4. The chart

In [5]:
COLORS = {'in': '#4C72B0', 'out': '#DD8452', 'storage': '#55A868'}

def plot_waterfall(ax, wb, title):
    terms, amounts, wb_start, wb_end, kinds = waterfall_geometry(wb)
    for i, (term, amt, s, e, kind) in enumerate(zip(terms, amounts, wb_start, wb_end, kinds)):
        lo, hi = min(s, e), max(s, e)
        ax.add_patch(mpatches.Rectangle((i - 0.45, lo), 0.9, hi - lo, color=COLORS[kind]))
        ax.text(i, (s + e) / 2, f'{abs(amt):.0f}', ha='center', va='center', fontsize=8)
    ax.set_xticks(range(len(terms)))
    ax.set_xticklabels([t.replace(' ', '\n') for t in terms], fontsize=8)
    ax.set_ylabel('Amount (mm)')
    ax.set_ylim(-20, 1100)
    ax.set_title(title, fontsize=11)
    handles = [mpatches.Patch(color=c, label=l) for l, c in
               [('in', COLORS['in']), ('out', COLORS['out']), ('storage', COLORS['storage'])]]
    ax.legend(handles=handles, loc='upper right', fontsize=8, framealpha=0.9)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_waterfall(axes[0], driest, 'A: Driest period')
plot_waterfall(axes[1], wettest, 'B: Wettest period')
fig.suptitle('Urban catchment water balance (Mitchell et al. 2003)', fontsize=12)
plt.tight_layout()
plt.savefig('../../images/2026-09_water-balance-waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size ... with Axes>

## References

- Ladson, A.R. (2017). [Graphing a water balance](https://tonyladson.wordpress.com/2017/08/15/graphing-a-water-balance/). R source: [gist.github.com/TonyLadson/4d42e2cedc20aa1ff04a06631af88551](https://gist.github.com/TonyLadson/4d42e2cedc20aa1ff04a06631af88551).
- Mitchell, V.G., McMahon, T.A. & Mein, R.G. (2003). Components of the Total Water Balance of an Urban Catchment. *Environmental Management* 32(6): 735-746.